# ufakzeka on Colab

Runs SFT, DPO and evaluation for ufakzeka-1 on a Colab GPU. Data and checkpoints live on the Modal volumes; this notebook pulls what it needs and pushes results back.

Before running: Runtime > Change runtime type > GPU (A100 or L4 with Pro; T4 works too, about 3x slower). Add three secrets in the key icon on the left: `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `HF_TOKEN`, with notebook access enabled.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install modal tokenizers 'transformers>=4.45' datasets zstandard 2>&1 | tail -1

In [ ]:
import os
from google.colab import userdata
os.environ['MODAL_TOKEN_ID'] = userdata.get('MODAL_TOKEN_ID')
os.environ['MODAL_TOKEN_SECRET'] = userdata.get('MODAL_TOKEN_SECRET')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!modal profile current

## Code and data from the Modal volumes

In [ ]:
%%bash
set -e
mkdir -p /content/w /data/tokenizer /data/sft /ckpt
cd /content/w
modal volume get ufakzeka-data code/ufakzeka-repo.tgz repo.tgz --force >/dev/null
tar xzf repo.tgz && rm repo.tgz
modal volume get ufakzeka-data tokenizer/ufakzeka.json /data/tokenizer/ufakzeka.json --force >/dev/null
for f in train.bin train.mask val.bin val.mask; do modal volume get ufakzeka-data sft/$f /data/sft/$f --force >/dev/null; done
ls /content/w /data/sft
du -sh /data/sft

Set `BASE` to the checkpoint to start from: `ufakzeka-1/final.pt` (stage 1) or `ufakzeka-1-s2/final.pt` (after stage 2).

In [ ]:
BASE = 'ufakzeka-1-s2/final.pt'   # change if needed
NAME = 'ufakzeka-1-instruct-v4'
import subprocess, os
os.makedirs('/ckpt/' + BASE.split('/')[0], exist_ok=True)
subprocess.run(['modal','volume','get','ufakzeka-ckpt',BASE,'/ckpt/'+BASE,'--force'], check=True)
print(os.path.getsize('/ckpt/'+BASE)/1e6, 'MB')

## SFT

In [ ]:
%cd /content/w
!python -m ufakzeka.train.sft_run --base /ckpt/{BASE} --out /ckpt/{NAME} --data /data/sft --tokenizer /data/tokenizer/ufakzeka.json --epochs 3 --micro-batch 8

## DPO (starts from the SFT result above)

In [ ]:
%cd /content/w
!python -m ufakzeka.train.dpo_run --sft /ckpt/{NAME}/final.pt --out /ckpt/{NAME}-dpo --tokenizer /data/tokenizer/ufakzeka.json

## Quick chat test

In [ ]:
%cd /content/w
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
path = f'/ckpt/{NAME}-dpo/hf'
tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(path, trust_remote_code=True, dtype=torch.float32).cuda().eval()
end = tok.convert_tokens_to_ids('<|im_end|>')
def ask(q):
    ids = tok(f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n', return_tensors='pt').input_ids.cuda()
    out = model.generate(ids, max_new_tokens=100, do_sample=True, temperature=0.5, top_p=0.9, repetition_penalty=1.05, eos_token_id=[end, 0], pad_token_id=3)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
for q in ['merhaba', 'kimsin', 'Türkiye\'nin başkenti neresi', '45 artı 27 kaç', 'şu anda saat kaç', 'İstanbul nerede']:
    print(q, '->', ask(q))

## Push results back to the Modal volume

In [ ]:
import subprocess
for n in [NAME, NAME + '-dpo']:
    subprocess.run(['modal','volume','put','ufakzeka-ckpt', f'/ckpt/{n}', n, '--force'], check=True)
    print('uploaded', n)

## Optional: benchmark (about 25 min on L4, 12 on A100)

In [ ]:
%cd /content/w
!pip -q install 'lm_eval>=0.4.9' accelerate 2>&1 | tail -1
!lm_eval --model hf --model_args pretrained=/ckpt/{NAME}-dpo/hf,dtype=bfloat16,trust_remote_code=True --tasks hellaswag_tr,arc_tr_challenge,arc_tr_easy,xcopa_tr,belebele_tur_Latn,turblimp_core,turkishmmlu --include_path ufakzeka/eval/tasks --batch_size 16 --trust_remote_code 2>&1 | grep -E '^\|' | grep -vE 'Filter|^\|-|turblimp_[a-z]|turkishmmlu_'